# Failure Prediction - Exploratory Data Analysis

This notebook explores the SCADA dataset behind the **Failure Prediction Module** and
documents the reasoning behind the target definition, the feature design and the
validation strategy.

**Production logic lives in `src/wind_turbine_pm/`.** This notebook imports and calls
those functions rather than reimplementing them, so anything shown here is exactly what
the training pipeline and the serving path do.

> **Data disclosure:** unless `configs/data.yaml -> data.source` has been repointed, the
> dataset is **synthetic** - generated by this project, not measured from real turbines.
> Every number below describes simulated data.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from wind_turbine_pm.config import load_config
from wind_turbine_pm.constants import FAILURE_EVENT, TARGET_COLUMN, TIMESTAMP, TURBINE_ID
from wind_turbine_pm.data.ingestion import is_synthetic, load_raw_dataset
from wind_turbine_pm.data.preprocessing import apply_modelling_filter, create_failure_target, preprocess
from wind_turbine_pm.data.splitting import compute_boundaries
from wind_turbine_pm.data.validation import validate_scada_frame
from wind_turbine_pm.features.failure_features import build_failure_features
from wind_turbine_pm.logging_config import configure_logging

configure_logging(level="WARNING")
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

cfg = load_config()
print("Data source:", cfg.require("data.source"), "| synthetic:", is_synthetic(cfg))

## 1. Dataset overview

In [ ]:
raw = load_raw_dataset(cfg)

print(f"Rows:            {len(raw):,}")
print(f"Columns:         {raw.shape[1]}")
print(f"Turbines:        {raw[TURBINE_ID].nunique()}")
print(f"Time coverage:   {raw[TIMESTAMP].min()}  ->  {raw[TIMESTAMP].max()}")
print(f"Span:            {(raw[TIMESTAMP].max() - raw[TIMESTAMP].min()).days} days")
print(f"Failure events:  {int(raw[FAILURE_EVENT].sum())}")
raw.head()

In [ ]:
raw.describe().T[["count", "mean", "std", "min", "50%", "max"]].round(2)

## 2. Data validation

The raw file deliberately contains injected defects - missing values, physically
impossible readings, duplicate rows and a few records with no key - so the validation
layer has something real to detect. This is what the pipeline reports before any
cleaning happens.

In [ ]:
report = validate_scada_frame(raw, cfg)

print(f"Valid: {report.is_valid} | errors: {len(report.errors)} | warnings: {len(report.warnings)}")
pd.DataFrame(
    [
        {"check": f.check, "severity": str(f.severity), "count": f.count, "message": f.message[:90]}
        for f in report.findings
    ]
)

### Missing-value analysis

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, max(3, 0.3 * len(missing))))
ax.barh(missing.index[::-1], missing.to_numpy()[::-1] * 100, color="#2b6cb0")
ax.set_xlabel("Missing (%)")
ax.set_title("Missingness by column (raw data)")
plt.tight_layout()
plt.show()

## 3. Cleaning and target construction

`preprocess` parses timestamps, drops unusable rows, de-duplicates, replaces
out-of-range readings with NaN and imputes using **past-only** information within each
turbine.

`create_failure_target` then attaches `failure_within_48h`: for observation *t* of
turbine *k* the label is 1 when a failure occurs for that same turbine in the half-open
interval `(t, t + 48h]`.

In [ ]:
clean, stats = preprocess(raw, cfg)
print("Preprocessing:", stats)

labelled = create_failure_target(clean, cfg)
eligible = apply_modelling_filter(labelled)

print(f"\nRows after cleaning:   {len(clean):,}")
print(f"Rows eligible for ML:  {len(eligible):,}")
print(f"Dropped as ineligible: {len(labelled) - len(eligible):,}")

### Class imbalance

In [ ]:
positive_rate = eligible[TARGET_COLUMN].mean()
counts = eligible[TARGET_COLUMN].value_counts().sort_index()

print(f"Negative (0): {counts.get(0, 0):,}")
print(f"Positive (1): {counts.get(1, 0):,}")
print(f"Positive rate: {positive_rate:.3%}  (roughly 1 in {int(1 / positive_rate)})")
print(f"\nA model predicting 'no failure' everywhere would score {1 - positive_rate:.2%} accuracy.")
print("This is why accuracy is never used as the headline metric for this module.")

fig, ax = plt.subplots(figsize=(5, 3.4))
ax.bar(["no failure", "failure within 48h"], counts.to_numpy(), color=["#2f855a", "#c53030"])
ax.set_yscale("log")
ax.set_ylabel("Observations (log scale)")
ax.set_title("Class balance")
plt.tight_layout()
plt.show()

### Failure frequency per turbine

In [ ]:
per_turbine = (
    labelled.groupby(TURBINE_ID)
    .agg(observations=(TIMESTAMP, "size"), failures=(FAILURE_EVENT, "sum"), positive_rate=(TARGET_COLUMN, "mean"))
    .sort_values("failures", ascending=False)
)
display(per_turbine.round(4))

fig, ax = plt.subplots()
ax.bar(per_turbine.index, per_turbine["failures"], color="#c53030")
ax.set_ylabel("Failure events")
ax.set_title("Failure events per turbine")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

Turbines differ in how often they fail. A turbine with **no** failures at all is a
realistic case the validation layer warns about but does not treat as fatal - it still
contributes negative examples and its own baseline behaviour.

## 4. Sensor distributions and physical relationships

In [ ]:
sensors = ["wind_speed", "power_output", "rotor_speed", "gearbox_temperature", "vibration", "oil_pressure"]

fig, axes = plt.subplots(2, 3, figsize=(14, 6.5))
for ax, sensor in zip(axes.ravel(), sensors):
    ax.hist(clean[sensor].dropna(), bins=60, color="#2b6cb0", alpha=0.85)
    ax.set_title(sensor)
plt.suptitle("Sensor distributions (after cleaning)")
plt.tight_layout()
plt.show()

### Power curve

In [ ]:
sample = clean.sample(min(20000, len(clean)), random_state=0)
running = sample[sample["operational_status"] == "normal"]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.scatter(running["wind_speed"], running["power_output"], s=3, alpha=0.25, color="#2b6cb0")
ax.set_xlabel("Wind speed (m/s)")
ax.set_ylabel("Power output (kW)")
ax.set_title("Observed power curve (normal operation)")
plt.tight_layout()
plt.show()

print("Power rises roughly with the cube of wind speed up to rated wind speed,")
print("then holds flat at rated power - the expected pitch-regulated behaviour.")
print(f"corr(wind_speed, power_output) = {running['wind_speed'].corr(running['power_output']):.3f}")

In [ ]:
checks = {
    "wind -> power": ("wind_speed", "power_output"),
    "rotor -> generator speed": ("rotor_speed", "generator_speed"),
    "load -> gearbox temperature": ("power_output", "gearbox_temperature"),
    "ambient -> nacelle temperature": ("ambient_temperature", "nacelle_temperature"),
    "rotor speed -> vibration": ("rotor_speed", "vibration"),
}
pd.DataFrame(
    [{"relationship": name, "correlation": round(running[a].corr(running[b]), 3)} for name, (a, b) in checks.items()]
)

## 5. Failure versus non-failure comparison

In [ ]:
compare = ["vibration", "gearbox_temperature", "bearing_temperature", "generator_temperature", "oil_pressure", "oil_temperature"]

summary = eligible.groupby(TARGET_COLUMN)[compare].mean().T
summary.columns = ["no failure", "failure within 48h"]
summary["difference"] = summary["failure within 48h"] - summary["no failure"]
summary["relative %"] = (summary["difference"] / summary["no failure"].abs() * 100).round(2)
summary.round(3)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 6.5))
for ax, sensor in zip(axes.ravel(), compare):
    for label, colour, name in ((0, "#2f855a", "no failure"), (1, "#c53030", "failure within 48h")):
        values = eligible.loc[eligible[TARGET_COLUMN] == label, sensor].dropna()
        ax.hist(values, bins=50, density=True, alpha=0.5, color=colour, label=name)
    ax.set_title(sensor)
axes[0, 0].legend(fontsize=8)
plt.suptitle("Sensor distributions by target class")
plt.tight_layout()
plt.show()

The distributions **overlap heavily**. That is deliberate and important: no single
sensor threshold separates the classes, so the model has to learn a multivariate,
temporal pattern. A dataset where one feature cleanly split the target would make the
whole pipeline look far better than it is.

## 6. Pre-failure trends

In [ ]:
if "hours_to_failure" in labelled.columns:
    window = labelled[labelled["hours_to_failure"].between(0, 240)].copy()
    window["hours_bucket"] = (window["hours_to_failure"] // 12 * 12).astype(int)

    trend_sensors = ["vibration", "gearbox_temperature", "bearing_temperature", "oil_pressure"]
    fig, axes = plt.subplots(2, 2, figsize=(13, 6))
    for ax, sensor in zip(axes.ravel(), trend_sensors):
        profile = window.groupby("hours_bucket")[sensor].mean()
        ax.plot(profile.index, profile.to_numpy(), marker="o", ms=3.5, color="#c53030")
        ax.invert_xaxis()
        ax.axvline(48, ls="--", c="black", lw=1, label="48h prediction horizon")
        ax.set_xlabel("Hours before failure")
        ax.set_title(sensor)
    axes[0, 0].legend(fontsize=8)
    plt.suptitle("Average sensor behaviour approaching a failure")
    plt.tight_layout()
    plt.show()
else:
    print("hours_to_failure is only produced by the synthetic generator.")

Degradation develops **gradually** over days, and the signal inside the 48-hour horizon
is meaningful but far from a step change. Vibration and drivetrain temperatures drift
up, oil pressure drifts down - which is what motivates the rolling, slope and
deviation-from-baseline features.

## 7. Example turbine timeline

In [ ]:
worst = labelled.groupby(TURBINE_ID)[FAILURE_EVENT].sum().idxmax()
one = labelled[labelled[TURBINE_ID] == worst].sort_values(TIMESTAMP)
failure_times = one.loc[one[FAILURE_EVENT] == 1, TIMESTAMP]

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
for ax, sensor in zip(axes, ["vibration", "gearbox_temperature", "oil_pressure", "power_output"]):
    ax.plot(one[TIMESTAMP], one[sensor], lw=0.7, color="#2b6cb0")
    for moment in failure_times:
        ax.axvline(moment, color="#c53030", ls="--", lw=1.2)
    ax.set_ylabel(sensor, fontsize=9)

axes[-1].set_xlabel("Time")
plt.suptitle(f"Turbine {worst} - red lines mark failure events")
plt.tight_layout()
plt.show()

## 8. Correlation structure

In [ ]:
numeric = [c for c in cfg.require("features.raw_sensors") if c in clean.columns]
corr = clean[numeric].corr()

fig, ax = plt.subplots(figsize=(9.5, 8))
image = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(numeric)), numeric, rotation=90, fontsize=8)
ax.set_yticks(range(len(numeric)), numeric, fontsize=8)
fig.colorbar(image, shrink=0.8)
ax.set_title("Sensor correlation matrix")
plt.tight_layout()
plt.show()

## 9. Feature engineering

`build_failure_features` is the single entry point used by training, the API and the
dashboard. Every temporal operation is grouped by `turbine_id` and reads only current or
past rows.

In [ ]:
features, spec = build_failure_features(labelled, cfg)
features = features.loc[eligible.index]

print(f"Features generated: {len(spec.names)}")
pd.DataFrame(
    [{"group": name, "n_features": len(members), "example": members[0] if members else ""} for name, members in spec.groups.items()]
)

In [ ]:
target = eligible[TARGET_COLUMN]
correlations = (
    features.corrwith(target)
    .dropna()
    .abs()
    .sort_values(ascending=False)
    .head(25)
)

fig, ax = plt.subplots(figsize=(9, 6.5))
ax.barh(correlations.index[::-1], correlations.to_numpy()[::-1], color="#2b6cb0")
ax.set_xlabel("|correlation| with failure_within_48h")
ax.set_title("Top 25 features by univariate correlation with the target")
plt.tight_layout()
plt.show()

### Target-leakage discussion

The strongest univariate correlations above are around 0.1-0.3, not 0.9. **That is the
expected and desired result.** A feature correlating near-perfectly with the target
would be a leak, not a discovery.

Concretely, the pipeline defends against leakage in six ways:

1. **Grouping.** Every lag, rolling window, slope and expanding baseline is computed
   inside `groupby("turbine_id")`, so one turbine's data can never inform another's.
2. **Trailing windows only.** Rolling statistics end at the current row. Nothing reads
   an index greater than the row being computed.
3. **Shifted baselines.** Expanding means/medians are shifted by one row, so an
   observation never contributes to the baseline it is compared against.
4. **No backward fill.** Imputation forward-fills (copying the *past* forward) and
   otherwise uses a robust per-turbine statistic. Backward filling would copy a future
   reading into the present.
5. **Excluded columns.** `failure_event`, `maintenance_event`, `degradation_level`,
   `hours_to_failure`, `episode_id` and `failure_mode` are ground-truth columns that
   exist for EDA only. They are listed in `features.exclude_columns` and can never enter
   the model matrix.
6. **Excluded rows.** Observations recorded during `fault`/`maintenance` are dropped:
   at that point the failure is already known, so keeping them would inflate measured
   performance.

The test suite pins these properties down empirically - `tests/test_failure_features.py`
perturbs the final observation and asserts that no earlier feature value moves, and
perturbs one turbine and asserts that no other turbine's features move.

## 10. Temporal split

In [ ]:
boundaries = compute_boundaries(eligible, cfg)
print("Split boundaries:")
for key, value in boundaries.to_dict().items():
    print(f"  {key:<14} {value}")

times = pd.to_datetime(eligible[TIMESTAMP])
parts = {
    "train": times <= boundaries.train_end,
    "valid": (times >= boundaries.valid_start) & (times <= boundaries.valid_end),
    "test": times >= boundaries.test_start,
}
rows = []
for name, mask in parts.items():
    subset = eligible.loc[mask.to_numpy()]
    rows.append({
        "split": name,
        "rows": len(subset),
        "positives": int(subset[TARGET_COLUMN].sum()),
        "positive_rate": round(float(subset[TARGET_COLUMN].mean()), 5),
        "start": subset[TIMESTAMP].min(),
        "end": subset[TIMESTAMP].max(),
    })
pd.DataFrame(rows)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 2.6))
colours = {"train": "#2b6cb0", "valid": "#b7791f", "test": "#2f855a"}
for name, mask in parts.items():
    subset = eligible.loc[mask.to_numpy()]
    ax.barh(0, subset[TIMESTAMP].max() - subset[TIMESTAMP].min(), left=subset[TIMESTAMP].min(),
            height=0.5, color=colours[name], label=name)
ax.axvline(boundaries.train_end, c="black", ls="--", lw=1)
ax.axvline(boundaries.valid_end, c="black", ls="--", lw=1)
ax.set_yticks([])
ax.set_title(f"Chronological split with a {boundaries.embargo_hours:.0f}h embargo at each boundary")
ax.legend(loc="upper center", ncol=3)
plt.tight_layout()
plt.show()

The **embargo** is the key detail. The label looks 48 hours forward, so an observation
just before a boundary carries information determined by events on the other side of it.
Removing a 48-hour band at each boundary makes that impossible. `compute_boundaries`
refuses to run if the configured embargo is shorter than the target horizon.

## 11. Summary

- The dataset is **synthetic**, multi-turbine, hourly, with realistic operating states,
  thermal inertia and gradual degradation.
- The target is a genuinely **rare event** (~2%), so PR-AUC, recall and F2 - not
  accuracy - are the metrics that matter.
- Class distributions **overlap**, so the problem is learnable but not trivial.
- Degradation develops over **days**, which justifies the multi-scale rolling and trend
  features.
- Leakage is prevented structurally and verified by tests.

Next steps are in the pipeline scripts:

```bash
python scripts/train_failure_model.py
python scripts/evaluate_failure_model.py
```